In [6]:
import json

#jsonl_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes.jsonl"  # adjust path
jsonl_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"  # adjust path

with open(jsonl_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Line {i}: {e}")
            print(f"  Content: {line[:400]}")  # show first 400 chars

Line 2366: Expecting ',' delimiter: line 1 column 329 (char 328)
  Content: {"script_id": "synthetic_script_0681_error", "scene_id": "synthetic_script_0681_error_scene_02", "scene_index": 2, "slugline": "INT. TECH LAB - CONT'D", "location": "TECH LAB - CONT'D", "time_of_day": null, "scene_text": "The crowd gasps as a screen flickers to life. The AI, named AIDEN, appears on the screen. AIDEN Hello, I ": "DAY", "scene_text": "Mark and Jennifer meet regularly at the coffee s
Line 5505: Expecting ',' delimiter: line 1 column 681 (char 680)
  Content: {"script_id": "synthetic_script_0287_error", "scene_id": "synthetic_script_0287_error_scene_08", "scene_index": 8, "slugline": "INT. JESSICA'S APARTMENT - NIGHT", "location": "JESSICA'S APARTMENT", "time_of_day": "NIGHT", "scene_text": "Jessica sits at her computer, typing away at a new poem. She feels a renewed sense of identity and purpose, knowing that her past doesn't define her, but rather, i
Line 7055: Expecting ',' delimiter: line 1 col

In [1]:
import json

input_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"

# Extract the raw content at the exact byte positions from the error output
bad_positions = [3145313, 7339075, 9435876, 10867480]

with open(input_path, "r", encoding="utf-8") as f:
    content = f.read()

for pos in bad_positions:
    print(f"\n=== Position {pos} ===")
    snippet = content[pos:pos+500]
    print(repr(snippet))


=== Position 3145313 ===
'{"script_id": "synthetic_script_0681_error", "scene_id": "synthetic_script_0681_error_scene_02", "scene_index": 2, "slugline": "INT. TECH LAB - CONT\'D", "location": "TECH LAB - CONT\'D", "time_of_day": null, "scene_text": "The crowd gasps as a screen flickers to life. The AI, named AIDEN, appears on the screen. AIDEN Hello, I ": "DAY", "scene_text": "Mark and Jennifer meet regularly at the coffee shop to discuss her progress on the manuscript. They laugh, share stories, and grow closer with each '

=== Position 7339075 ===
'{"script_id": "synthetic_script_0287_error", "scene_id": "synthetic_script_0287_error_scene_08", "scene_index": 8, "slugline": "INT. JESSICA\'S APARTMENT - NIGHT", "location": "JESSICA\'S APARTMENT", "time_of_day": "NIGHT", "scene_text": "Jessica sits at her computer, typing away at a new poem. She feels a renewed sense of identity and purpose, knowing that her past doesn\'t define her, but rather, inspires her.", "characters": [], "props

In [2]:
snippet = content[9435876:9435876+1500]
print(snippet)  # without repr() to see actual characters

{"script_id": "synthetic_script_0043_error", "scene_id": "synthetic_script_0043_error_scene_09", "scene_index": 9, "slugline": "INT. COFFEE SHOP - DAY", "location": "COFFEE SHOP", "time_of_day": "DAY", "scene_text": "Dana approaches Dan at the counter. They exchange awkward pleasantries.\nDAN: So, you wanted to see me?\nDANA: Yes, Dan. I'm sorry for the way things ended between us. I've been thinking a lot about forgiveness lately.\nDan looks surprised but intrigued.\nDAN: (pausing) And what do you mean by that?\nDANA: I mean, I want to make things right between us. I know I hurt you, and I'm willing to face the consequences of my actions. I want to understand where you're coming from and make things right.\nDan looks at her for a long moment before nodding.\nDAN: Alright, Dana. Let's talk.\nTO BE CONTINUED...\nNote: This is a modern psychological screenplay, with a theme of forgiveness. It follows the r, "wardrobe": {}, "action_summary": "Jessica sits at her computer, typing away at a

In [3]:
snippet = content[9435876:9435876+1500]
print(snippet)  # without repr() to see actual characters

{"script_id": "synthetic_script_0043_error", "scene_id": "synthetic_script_0043_error_scene_09", "scene_index": 9, "slugline": "INT. COFFEE SHOP - DAY", "location": "COFFEE SHOP", "time_of_day": "DAY", "scene_text": "Dana approaches Dan at the counter. They exchange awkward pleasantries.\nDAN: So, you wanted to see me?\nDANA: Yes, Dan. I'm sorry for the way things ended between us. I've been thinking a lot about forgiveness lately.\nDan looks surprised but intrigued.\nDAN: (pausing) And what do you mean by that?\nDANA: I mean, I want to make things right between us. I know I hurt you, and I'm willing to face the consequences of my actions. I want to understand where you're coming from and make things right.\nDan looks at her for a long moment before nodding.\nDAN: Alright, Dana. Let's talk.\nTO BE CONTINUED...\nNote: This is a modern psychological screenplay, with a theme of forgiveness. It follows the r, "wardrobe": {}, "action_summary": "Jessica sits at her computer, typing away at a

In [4]:
import json
import re

input_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"
output_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes_enriched_fixed.jsonl"

with open(input_path, "r", encoding="utf-8") as f:
    content = f.read()

decoder = json.JSONDecoder()
scenes = []
skipped_raw = []
pos = 0
content = content.strip()

while pos < len(content):
    # Skip whitespace
    while pos < len(content) and content[pos] in ' \t\r\n':
        pos += 1
    if pos >= len(content):
        break
    try:
        obj, end_idx = decoder.raw_decode(content, pos)
        scenes.append(obj)
        pos = end_idx
    except json.JSONDecodeError as e:
        # Find the next { to try recovering the next object
        next_pos = content.find('\n{', pos)
        if next_pos == -1:
            break
        skipped_raw.append(content[pos:next_pos])
        print(f"[SKIP] At pos {pos}: {repr(content[pos:pos+80])}")
        pos = next_pos + 1  # skip to next line starting with {

print(f"\nLoaded: {len(scenes)} scenes")
print(f"Skipped fragments: {len(skipped_raw)}")

# Write fixed file
with open(output_path, "w", encoding="utf-8") as f_out:
    for scene in scenes:
        f_out.write(json.dumps(scene, ensure_ascii=False) + "\n")

print(f"Written: {len(scenes)} scenes to {output_path}")

[SKIP] At pos 3145313: '{"script_id": "synthetic_script_0681_error", "scene_id": "synthetic_script_0681_'
[SKIP] At pos 7339075: '{"script_id": "synthetic_script_0287_error", "scene_id": "synthetic_script_0287_'
[SKIP] At pos 9435876: '{"script_id": "synthetic_script_0043_error", "scene_id": "synthetic_script_0043_'
[SKIP] At pos 10867480: 'ules of proper screenplay format, including scene headers, character dialogue in'

Loaded: 10808 scenes
Skipped fragments: 4
Written: 10808 scenes to /home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/train_error_scenes_enriched_fixed.jsonl


In [5]:
import json
import re

input_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/test_error_scenes_enriched.jsonl"
output_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/test_error_scenes_enriched_fixed.jsonl"

with open(input_path, "r", encoding="utf-8") as f:
    content = f.read()

decoder = json.JSONDecoder()
scenes = []
skipped_raw = []
pos = 0
content = content.strip()

while pos < len(content):
    # Skip whitespace
    while pos < len(content) and content[pos] in ' \t\r\n':
        pos += 1
    if pos >= len(content):
        break
    try:
        obj, end_idx = decoder.raw_decode(content, pos)
        scenes.append(obj)
        pos = end_idx
    except json.JSONDecodeError as e:
        # Find the next { to try recovering the next object
        next_pos = content.find('\n{', pos)
        if next_pos == -1:
            break
        skipped_raw.append(content[pos:next_pos])
        print(f"[SKIP] At pos {pos}: {repr(content[pos:pos+80])}")
        pos = next_pos + 1  # skip to next line starting with {

print(f"\nLoaded: {len(scenes)} scenes")
print(f"Skipped fragments: {len(skipped_raw)}")

# Write fixed file
with open(output_path, "w", encoding="utf-8") as f_out:
    for scene in scenes:
        f_out.write(json.dumps(scene, ensure_ascii=False) + "\n")

print(f"Written: {len(scenes)} scenes to {output_path}")


Loaded: 774 scenes
Skipped fragments: 0
Written: 774 scenes to /home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/test_error_scenes_enriched_fixed.jsonl


In [1]:
import json
import re

input_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/val_error_scenes_enriched.jsonl"
output_path = "/home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/val_error_scenes_enriched_fixed.jsonl"

with open(input_path, "r", encoding="utf-8") as f:
    content = f.read()

decoder = json.JSONDecoder()
scenes = []
skipped_raw = []
pos = 0
content = content.strip()

while pos < len(content):
    # Skip whitespace
    while pos < len(content) and content[pos] in ' \t\r\n':
        pos += 1
    if pos >= len(content):
        break
    try:
        obj, end_idx = decoder.raw_decode(content, pos)
        scenes.append(obj)
        pos = end_idx
    except json.JSONDecodeError as e:
        # Find the next { to try recovering the next object
        next_pos = content.find('\n{', pos)
        if next_pos == -1:
            break
        skipped_raw.append(content[pos:next_pos])
        print(f"[SKIP] At pos {pos}: {repr(content[pos:pos+80])}")
        pos = next_pos + 1  # skip to next line starting with {

print(f"\nLoaded: {len(scenes)} scenes")
print(f"Skipped fragments: {len(skipped_raw)}")

# Write fixed file
with open(output_path, "w", encoding="utf-8") as f_out:
    for scene in scenes:
        f_out.write(json.dumps(scene, ensure_ascii=False) + "\n")

print(f"Written: {len(scenes)} scenes to {output_path}")


Loaded: 1170 scenes
Skipped fragments: 0
Written: 1170 scenes to /home/sagemaker-user/script-supervisor-ai/datasets/scenes_enriched_error/val_error_scenes_enriched_fixed.jsonl
